In [3]:
from src.features import fit_feature_pipeline, transform_features, make_Xy, CAT_COLS, KEEP_INDICATORS, EXCLUDE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix, classification_report
from xgboost import XGBClassifier
import joblib
import os
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import shap
import gc

ModuleNotFoundError: No module named 'pandas'

# Merging the training data

In [ ]:
train_tx = pd.read_csv("data/train_transaction.csv")

print("Shape" , train_tx.shape)
print("Memory used(MB)", round(train_tx.memory_usage(deep= True).sum()/ 1024**2,1))

train_tx.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/train_transaction.csv'

In [ ]:
def reduce_memory_size(df):
    "Downsizing the numeric columns to smallest safe dtype to save memory but ensure to not lose any information"
    for col in df.columns:
        col_type = df[col].dtype
        if col_type == "float64":
            df[col] = pd.to_numeric(df[col], downcast ="float")
        elif col_type == "int64":
            df[col] = pd.to_numeric(df[col], downcast ="integer")

    return df

train_tx = reduce_memory_size(train_tx)

print("Memory used after downcasting(MB)", round(train_tx.memory_usage(deep= True).sum()/ 1024**2,1))

In [ ]:
train_id = pd.read_csv("data/train_identity.csv")

print("Shape" , train_id.shape)
print("Memory used(MB)", round(train_id.memory_usage(deep= True).sum()/ 1024**2,1))

In [ ]:
train_id = reduce_memory_size(train_id)

print("Shape" , train_tx.shape)
print("Shape" , train_id.shape)

In [ ]:
train_data = train_tx.merge(train_id, on = "TransactionID", how ="left")

print("Merged shape:", train_data.shape)
print("Rows match transaction count:", train_data.shape[0] == train_tx.shape[0])

In [ ]:
train_data.to_parquet("data/train_data_merged.parquet")

# Data Interrogation

In [ ]:
train_data = pd.read_parquet("data/train_data_merged.parquet")

print("shape", train_data.shape)

In [ ]:
# target variable
counts = train_data["isFraud"].value_counts()

print(counts)
print()
print("Fraud rate", round(train_data["isFraud"].mean()*100, 2), "%")
print("Imbalance Ratio", round(counts.loc[0]/counts.loc[1],1))

In [ ]:
# Missing data

missing_pct = train_data.isnull().mean() * 100
missing_pct = missing_pct.sort_values(ascending = False)

print("Columns >90% missing:", (missing_pct > 90).sum())
print("Columns 50-90% missing:", ((missing_pct > 50) & (missing_pct <= 90)).sum())
print("Columns 1-50% missing:", ((missing_pct > 1) & (missing_pct <= 50)).sum())
print("Columns with no missing:", (missing_pct == 0).sum())
print()
print("Top 15 most-missing columns:")
print(missing_pct.head(15).round(1))

In [ ]:
train_data["has_identity"] = train_data["id_31"].notnull().astype(int)

print(train_data.groupby("has_identity")["isFraud"].mean().round(4) * 100)
print()
print("Overall fraud rate:", round(train_data["isFraud"].mean() * 100, 2), "%")

In [ ]:
# Fraud rate by Product Type , Card Type and Card network
print("Fraud Rate by ProductCD (%)")

print((train_data.groupby("ProductCD")["isFraud"].mean() *100).round(2).sort_values(ascending = False))
print()

print("Fraud Rate by Card Type (%)")

print((train_data.groupby("card6")["isFraud"].mean() *100).round(2).sort_values(ascending = False))
print()

print("Fraud Rate by Card Network (%)")

print((train_data.groupby("card4")["isFraud"].mean() *100).round(2).sort_values(ascending = False))
print()

In [ ]:
# Count of the highest rated frauds

for col in ["ProductCD","card6","card4"]:
    print(f"\n{col}")
    summary = train_data.groupby(col)["isFraud"].agg(["mean","count"])
    summary["fraud_rate_%"] = (summary["mean"] * 100).round(2)
    summary["share_%"] = (summary["count"] / len(train_data) * 100).round(1)
    print(summary[["fraud_rate_%", "count", "share_%"]].sort_values("fraud_rate_%", ascending=False))

In [ ]:
# Transaction Amount

print("Transaction summary by amount")

print(train_data.groupby("isFraud")["TransactionAmt"].describe().round(2))
print()

print(" Median Amount - legit" , round(train_data[train_data["isFraud"] == 0]["TransactionAmt"].median(),2))
print(" Median Amount - fraud" , round(train_data[train_data["isFraud"] == 1]["TransactionAmt"].median(),2))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# log1p handles the skew; log(amount+1) so zero amount transactions are safe
ax.hist(np.log1p(train_data[train_data["isFraud"]==0]["TransactionAmt"]),
        bins=60, alpha=0.5, density=True, label="Legit")
ax.hist(np.log1p(train_data[train_data["isFraud"]==1]["TransactionAmt"]),
        bins=60, alpha=0.5, density=True, label="Fraud")

ax.set_xlabel("log(1 + TransactionAmt)")
ax.set_ylabel("Density")
ax.set_title("Transaction amount distribution: fraud vs legit")
ax.legend()
plt.show()

In [ ]:
# Time based classification of frauds

train_data["hour"] = (train_data["TransactionDT"] / 3600) % 24
train_data["hour"] = train_data["hour"].astype(int)

# Fraud rate by hour
hourly = train_data.groupby("hour")["isFraud"].agg(["mean", "count"])
hourly["fraud_rate_%"] = (hourly["mean"] * 100).round(2)

print(hourly[["fraud_rate_%", "count"]])

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

# Fraud rate line (left axis)
ax1.plot(hourly.index, hourly["fraud_rate_%"], color="crimson", marker="o", label="Fraud rate (%)")
ax1.set_xlabel("Hour (of the DT reference day)")
ax1.set_ylabel("Fraud rate (%)", color="crimson")
ax1.tick_params(axis="y", labelcolor="crimson")
ax1.axhline(3.5, color="crimson", linestyle="--", alpha=0.4, label="Baseline 3.5%")

# Transaction volume bars (right axis)
ax2 = ax1.twinx()
ax2.bar(hourly.index, hourly["count"], alpha=0.2, color="steelblue")
ax2.set_ylabel("Transaction count", color="steelblue")
ax2.tick_params(axis="y", labelcolor="steelblue")

ax1.set_title("Fraud rate vs transaction volume by hour")
ax1.set_xticks(range(0, 24))
fig.tight_layout()
plt.show()

# Split 

In [ ]:
train_data = pd.read_parquet("data/train_data_merged.parquet")

train_data["hour"] = ((train_data["TransactionDT"] / 3600) % 24).astype(int)

print("shape", train_data.shape)

In [ ]:
# TransactionDT is seconds from some reference point
span_days = (train_data["TransactionDT"].max() - train_data["TransactionDT"].min()) / (3600 * 24)
print("Time span (days):", round(span_days, 1))
print("Min DT:", train_data["TransactionDT"].min())
print("Max DT:", train_data["TransactionDT"].max())

In [ ]:
# Arrange in chronological order 
train_data = train_data.sort_values("TransactionDT").reset_index(drop=True)

# Slicing at 70% and 85% of rows 
n = len(train_data)
train_end = int(n*0.70)
val_end = int(n*0.85)

# Cutting the data into 3 parts 
train_set = train_data.iloc[:train_end]
val_set = train_data.iloc[train_end:val_end]
test_set = train_data.iloc[val_end:]

print("Train:", train_set.shape, "| rows 0 to", train_end)
print("Val:  ", val_set.shape,   "| rows", train_end, "to", val_end)
print("Test: ", test_set.shape,  "| rows", val_end, "to", n)

In [ ]:
for name, s in [("Train",train_set), ("Val", val_set), ("Test",test_set)]:
    lo = s["TransactionDT"].min()
    hi = s["TransactionDT"].max()

    print(f"{name:5} | DT {lo:>9} to {hi:>9} | days {lo/86400:6.1f} to {hi/86400:6.1f}")

In [ ]:
for name, s in [("Train",train_set), ("Val", val_set), ("Test",test_set)]:
    print(f"{name:5} | fraud rate: {s['isFraud'].mean() * 100 : .2f}% | n = {len(s):,}")

# Baseline Model

In [ ]:
# Baseline model with basic numerical features 
baseline_features = [
    "TransactionAmt", "hour", 
    "card1", "card2", "card3", "card5",
    "addr1", "addr2",
    "C1", "C2", "C5", "C13", "C14",
    "D1", "D4", "D10", "D15",
]

# Rebuild Hour in each split of dataset

for s in [train_set, val_set, test_set]:
    s["hour"] = ((s["TransactionDT"]/3600) %24).astype(int)


print("Missingness in baseline features (train):")

print((train_set[baseline_features].isnull().mean() *100).round(1))

In [ ]:
# X and Y for train and validation set
X_train = train_set[baseline_features].copy()
X_val = val_set[baseline_features].copy()
y_train = train_set["isFraud"]
y_val = val_set["isFraud"]

# Imputing missing values with median values 

imputer = SimpleImputer(strategy = "median")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.fit_transform(X_val)

# Scaling the features 

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_val_sc = scaler.fit_transform(X_val_imp)

# Logistic Regression Model

lr = LogisticRegression(max_iter = 100, class_weight ="balanced")
lr.fit(X_train_sc, y_train)

# Probabilities on Validation and score

val_probs = lr.predict_proba(X_val_sc)[:,1]

print("Baseline Logistic Regression — validation performance")
print("ROC-AUC:", round(roc_auc_score(y_val, val_probs), 4))
print("PR-AUC :", round(average_precision_score(y_val, val_probs), 4))

# Feature Engineering

In [ ]:
train_data = pd.read_parquet("data/train_data_merged.parquet")
train_data = train_data.sort_values("TransactionDT").reset_index(drop=True)

n = len(train_data)
train_set = train_data.iloc[:int(n*0.70)].copy()
val_set   = train_data.iloc[int(n*0.70):int(n*0.85)].copy()
test_set  = train_data.iloc[int(n*0.85):].copy()

# The categorical columns we want to bring in
cat_cols = ["ProductCD", "card4", "card6"]

for c in cat_cols:
    print(f"\n{c} — unique values and counts (train):")
    print(train_set[c].value_counts(dropna=False))

In [ ]:
# Encoding categorical variables

def encode_categoricals(df, cat_cols) :
    return pd.get_dummies(df, columns = cat_cols, dummy_na = True, dtype = int)

# encoding all theee sets
train_enc = encode_categoricals(train_set, cat_cols)
val_enc   = encode_categoricals(val_set, cat_cols)
test_enc  = encode_categoricals(test_set, cat_cols)

# Realigning columns across all sets

train_cols = train_enc.columns
val_enc = val_enc.reindex(columns = train_cols, fill_value = 0)
test_enc = test_enc.reindex(columns = train_cols, fill_value = 0)

new_cols = [c for c in train_enc.columns if any(c.startswith(p + "_") for p in cat_cols)]
print("New encoded columns")
print(new_cols)
print("\nShapes — train:", train_enc.shape, "| val:", val_enc.shape, "| test:", test_enc.shape)

In [ ]:
# Columns with substantial missingness 
missing_indicator_cols = [
    "addr1", "addr2",
    "D4", "D10", "D15", "D11", "D6", "D8", "D9", "D12", "D13", "D14",
    "dist1", "dist2",
    "id_31",
]

def add_missingness_indicators (df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[f"{c}_missing"] = df[c].isnull().astype(int)

    return df

train_enc = add_missingness_indicators(train_enc, missing_indicator_cols)
val_enc = add_missingness_indicators(val_enc, missing_indicator_cols)
train_enc = add_missingness_indicators(train_enc, missing_indicator_cols)

print("Fraud rate by the missingness indicator (train) : ")

for c in missing_indicator_cols : 
    ind = f"{c}_missing"
    if ind in train_enc.columns : 
        rates = train_enc.groupby(ind)["isFraud"].mean() * 100

        if 1 in rates.index and 0 in rates.index:
            print(f"  {c:8} | present: {rates.loc[0]:5.2f}%  missing: {rates.loc[1]:5.2f}%")

In [ ]:
# Keeping only indicators with significant difference in fraud , dropping the redundent clusters
keep_indicators = ["addr1", "D6", "D8", "D12", "dist2", "id_31"]
drop_indicators = ["addr2", "D4", "D10", "D15", "D11", "D9", "D13", "D14", "dist1"]

drop_cols = [f"{c}_missing" for c in drop_indicators]
for df_name, df in [("train", train_enc), ("val",val_enc),("test",test_enc)] :
    existing = [c for c in drop_cols if c in df.columns]
    df.drop(columns = existing, inplace = True)

print("Kept indicators", [f"{c}_missing" for c in keep_indicators])
print("Train shape now", train_enc.shape)

In [ ]:
# Card level aggregate

card_id = "card1"

# Learning about transactions mount of each card statistics 
card_stats = train_enc.groupby(card_id)["TransactionAmt"].agg(["mean","std"]).reset_index()
card_stats.columns = [card_id, "card_amt_mean", "card_amt_std"]

# Global fallbacks (for cards unseen in train, or std=0/NaN for single-transaction cards)
global_mean = train_enc["TransactionAmt"].mean()
global_std  = train_enc["TransactionAmt"].std()

def add_card_amt_features(df, stats, gmean, gstd):
    df = df.merge(stats, on=card_id, how="left")
    # Fill cards not seen in train with global stats
    df["card_amt_mean"] = df["card_amt_mean"].fillna(gmean)
    df["card_amt_std"]  = df["card_amt_std"].fillna(gstd)
    # Guard against zero/NaN std (cards with one transaction in train)
    df["card_amt_std"] = df["card_amt_std"].replace(0, gstd).fillna(gstd)
    # The z-score: how unusual is this amount FOR THIS CARD
    df["amt_z_for_card"] = (df["TransactionAmt"] - df["card_amt_mean"]) / df["card_amt_std"]
    return df

# 2. APPLY the train-learned stats to all three splits
train_enc = add_card_amt_features(train_enc, card_stats, global_mean, global_std)
val_enc   = add_card_amt_features(val_enc, card_stats, global_mean, global_std)
test_enc  = add_card_amt_features(test_enc, card_stats, global_mean, global_std)

# 3. Checking whether the new feature separate fraud better than raw amount did? (train only)
print("Mean amt_z_for_card by class (train):")
print(train_enc.groupby("isFraud")["amt_z_for_card"].mean().round(3))
print("\nMedian amt_z_for_card by class (train):")
print(train_enc.groupby("isFraud")["amt_z_for_card"].median().round(3))

In [ ]:
train_enc.to_parquet("data/train_features.parquet")
val_enc.to_parquet("data/val_features.parquet")
test_enc.to_parquet("data/test_features.parquet")
print("Feature splits saved.")

# Train Models

In [ ]:
train_enc = pd.read_parquet("data/train_features.parquet")
val_enc   = pd.read_parquet("data/val_features.parquet")
test_enc  = pd.read_parquet("data/test_features.parquet")

print("Train:", train_enc.shape, "| Val:", val_enc.shape, "| Test:", test_enc.shape)

In [ ]:
train_cols = set(train_enc.columns)
val_cols   = set(val_enc.columns)
test_cols  = set(test_enc.columns)

print("In train but NOT in test:")
print(sorted(train_cols - test_cols))
print("\nIn test but NOT in train:")
print(sorted(test_cols - train_cols))
print("\nIn train but NOT in val:")
print(sorted(train_cols - val_cols))
print("\nIn val but NOT in train:")
print(sorted(val_cols - train_cols))

In [ ]:
indicator_sources = {
    "addr1_missing": "addr1",
    "D6_missing": "D6",
    "D8_missing": "D8",
    "D12_missing": "D12",
    "dist2_missing": "dist2",
    "id_31_missing": "id_31",
}

# Rebuild them on test, using the same "is this null?" logic
for ind_col, src_col in indicator_sources.items():
    test_enc[ind_col] = test_enc[src_col].isnull().astype(int)

print("Test shape now:", test_enc.shape)

In [ ]:
train_cols = set(train_enc.columns)
print("Train == Val columns:", train_cols == set(val_enc.columns))
print("Train == Test columns:", train_cols == set(test_enc.columns))
print("All shapes — train:", train_enc.shape, "| val:", val_enc.shape, "| test:", test_enc.shape)

In [ ]:
test_enc.to_parquet("data/test_features.parquet")
print("Test features re-saved with schema fix.")

In [ ]:
# Excluding variables from feature set
exclude = ["isFraud", "TransactionDT", "TransactionID"]

feature_cols = [c for c in train_enc.columns if c not in exclude]

non_numeric = train_enc[feature_cols].select_dtypes(exclude=[np.number, "bool"]).columns.tolist()
print("Non-numeric feature columns still present:", non_numeric)
print("Total candidate features:", len(feature_cols))

In [ ]:
# Dropping non-numericals cols
numeric_features = [c for c in feature_cols if c not in non_numeric]

X_train = train_enc[numeric_features]
X_val = val_enc[numeric_features]
y_train = train_enc["isFraud"]
y_val = val_enc["isFraud"]

print("Feature count:", len(numeric_features))

# Imbalance handling using scale_pos_weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg/pos
print("scale pos weight", round(spw,1))

xgb = XGBClassifier(
    n_estimators = 400,
    learning_rate = 0.05,
    max_depth = 6,
    subsample = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight=spw,
    eval_metric="aucpr",
    n_jobs=-1,
    random_state=42,
)

xgb.fit(X_train,y_train)

# Scores on validation
val_probes = xgb.predict_proba(X_val)[:,1]
print("\n*XGBoost- validation performance")
print("ROC AUC Score", round(roc_auc_score(y_val,val_probes), 4))
print("Precision Score", round(average_precision_score(y_val, val_probes),4))
print("\n(Baseline LR : ROC-AUC 0.756 | PR-AUC 0.2236)")

In [ ]:
joblib.dump(xgb, "models/xgb_baseline.pkl")
print("Model saved.")

In [ ]:
train_enc = pd.read_parquet("data/train_features.parquet")
val_enc   = pd.read_parquet("data/val_features.parquet")
test_enc  = pd.read_parquet("data/test_features.parquet")


xgb = joblib.load("models/xgb_baseline.pkl")

print("Train:", train_enc.shape, "| Val:", val_enc.shape, "| Test:", test_enc.shape)

In [ ]:
# Reload RAW merged data and re-split (so the pipeline runs end-to-end from source)
raw = pd.read_parquet("data/train_data_merged.parquet").sort_values("TransactionDT").reset_index(drop=True)
n = len(raw)
tr = raw.iloc[:int(n*0.70)].copy()
va = raw.iloc[int(n*0.70):int(n*0.85)].copy()
te = raw.iloc[int(n*0.85):].copy()

# Fit on train, transform all splits
state = fit_feature_pipeline(tr)
tr_f = transform_features(tr, state)
va_f = transform_features(va, state)
te_f = transform_features(te, state)

X_train, y_train = make_Xy(tr_f, state)
X_val,   y_val   = make_Xy(va_f, state)
X_test,  y_test  = make_Xy(te_f, state)

print("Feature count:", len(state["feature_cols"]))
print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos

xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric="aucpr",
    n_jobs=-1,
    random_state=42,
)
xgb.fit(X_train, y_train)

val_probs = xgb.predict_proba(X_val)[:, 1]
print("XGBoost (consolidated pipeline) — validation:")
print("ROC-AUC:", round(roc_auc_score(y_val, val_probs), 4))
print("PR-AUC :", round(average_precision_score(y_val, val_probs), 4))
print("\n(Yesterday: ROC-AUC 0.9085 | PR-AUC 0.5423)")

In [ ]:
joblib.dump(xgb, "models/xgb_baseline.pkl")
print("Canonical model saved.")

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_val_sc   = scaler.transform(X_val_imp)
X_test_sc  = scaler.transform(X_test_imp)

print("Preprocessed for NN — train:", X_train_sc.shape, "| val:", X_val_sc.shape)
print("Any NaNs left in train?", np.isnan(X_train_sc).any())

In [ ]:
# Neural Network Deployment

torch.manual_seed(42)
device = torch.device("cpu")

# --- Convert to tensors ---
Xtr = torch.tensor(X_train_sc, dtype=torch.float32)
ytr = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
Xvl = torch.tensor(X_val_sc, dtype=torch.float32)
yvl = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=2048, shuffle=True)

# --- A simple MLP ---
class FraudMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x)

model = FraudMLP(Xtr.shape[1]).to(device)

# --- Imbalance handling: weight the positive class (NN equivalent of scale_pos_weight) ---
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# --- Training loop ---
n_epochs = 15
for epoch in range(1, n_epochs + 1):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    # Validation PR-AUC each epoch
    model.eval()
    with torch.no_grad():
        val_logits = model(Xvl)
        val_probs = torch.sigmoid(val_logits).numpy().ravel()
    pr = average_precision_score(y_val, val_probs)
    print(f"Epoch {epoch:2d} | val PR-AUC: {pr:.4f}")

roc = roc_auc_score(y_val, val_probs)
print(f"\nFinal — ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}")
print("(XGBoost bar: ROC-AUC 0.908 | PR-AUC 0.5419)")

In [ ]:
torch.save(model.state_dict(), "models/fraud_mlp.pt")
joblib.dump(imputer, "models/nn_imputer.pkl")
joblib.dump(scaler,  "models/nn_scaler.pkl")
print("Neural net + its preprocessing saved.")

# Probabilites to Decision and Interpretation

In [ ]:
# Rebuild splits + features from raw via the pipeline
raw = pd.read_parquet("data/train_data_merged.parquet").sort_values("TransactionDT").reset_index(drop=True)
n = len(raw)
tr = raw.iloc[:int(n*0.70)].copy()
va = raw.iloc[int(n*0.70):int(n*0.85)].copy()
te = raw.iloc[int(n*0.85):].copy()

state = fit_feature_pipeline(tr)
X_train, y_train = make_Xy(transform_features(tr, state), state)
X_val,   y_val   = make_Xy(transform_features(va, state), state)
X_test,  y_test  = make_Xy(transform_features(te, state), state)

# Load the canonical XGBoost
xgb = joblib.load("models/xgb_baseline.pkl")

print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)

# Sanity check: does XGBoost still score ~0.5419 on val?
val_probs = xgb.predict_proba(X_val)[:, 1]
print("XGBoost val PR-AUC:", round(average_precision_score(y_val, val_probs), 4), "(expected ~0.5419)")

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, val_probs)

# Showing the trade off at a range of thresholds

print(f"{'Thresh':>7} | {'Precision':>9} | {'Recall':>7} | {'Flagged':>8}")

print("-"*42)

for t in [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    predictions = (val_probs >= t).astype(int)

    tn,fp,fn,tp = confusion_matrix(y_val, predictions).ravel()
    prec = (tp)/(tp+fp) if (tp+fp) >0 else 0
    rec = tp/(tp+fn)
    print(f"{t:>7.2f} | {prec:>9.3f} | {rec:>7.3f} | {tp+fp:>8,}")

In [ ]:
# Cost Model : a missed fraud costs the transaction amount; a false flag costs a fixed amount

val_amounts = va["TransactionAmt"].values
FALSE_ALARM_COST = 5.0   # review labout + customer friction per flagged legit transaction

print(f"{'Thresh':>7} | {'Missed£':>12} | {'AlarmCost':>10} | {'TotalCost':>12} | {'Recall':>7}")
print("-"*60)

results = []
for t in np.arange(0.05, 1.00, 0.05):
    predictions = (val_probs >= t).astype(int)
    missed_fraud = (y_val.values == 1) & (predictions == 0)
    false_alarms = (y_val.values == 0) & (predictions == 1)

    missed_cost = val_amounts[missed_fraud].sum()
    alarm_cost = false_alarms.sum()* FALSE_ALARM_COST
    total = missed_cost + alarm_cost
    recall = ((y_val.values == 1) & (predictions == 1)).sum()/(y_val.values == 1).sum()
    results.append((t,total))
    print(f"{t:>7.2f} | {missed_cost:>12,.0f} | {alarm_cost:>10,.0f} | {total:>12,.0f} | {recall:>7.3f}")

best_t, best_cost = min(results, key= lambda r:r[1])
print(f"\nCost-minimizing threshold: {best_t:.2f}  (total cost {best_cost:,.0f})")

In [ ]:
print(f"{'AlarmCost':>10} | {'BestThresh':>10} | {'TotalCost':>12} | {'Recall':>7}")
print("-"*48)

for alarm_cost_assumption in [1,2,5,10,20,50,100]:
    best = None
    for t in np.arange(0.05,1.00,0.05):
        predictions = (val_probs>=t).astype(int)
        missed = (y_val.values == 1) & (predictions == 0)
        falarm = (y_val.values == 0) & (predictions == 1)
        total = val_amounts[missed].sum() + falarm.sum()*alarm_cost_assumption
        rec = ((y_val.values == 1) & (predictions == 1)).sum() / (y_val.values == 1).sum()
        if best is None or total < best[1]:
            best = (t, total, rec)

    print(f"{alarm_cost_assumption:>10} | {best[0]:>10.2f} | {best[1]:>12,.0f} | {best[2]:>7.3f}")

In [ ]:
CHOSEN_THRESHOLD = 0.30
predictions = (val_probs >= CHOSEN_THRESHOLD).astype(int)

tn,fp,fn,tp = confusion_matrix(y_val, predictions).ravel()

print(f"Operating point: threshold = {CHOSEN_THRESHOLD}\n")
print(f"Caught fraud (TP):        {tp:>7,}")
print(f"Missed fraud (FN):        {fn:>7,}")
print(f"False alarms (FP):        {fp:>7,}")
print(f"Correctly passed (TN):    {tn:>7,}")
print(f"\nOf {tp+fn:,} frauds, caught {tp/(tp+fn)*100:.1f}%")
print(f"Of {tp+fp:,} flagged, {tp/(tp+fp)*100:.1f}% were fraud")
print(f"Review load: {(tp+fp)/len(y_val)*100:.1f}% of all transactions")

In [ ]:
print(f"{'FlagBudget':>11} | {'Thresh':>7} | {'Recall':>7} | {'Precision':>9} | {'FraudCaught':>11}")
print("-" * 56)

for budget_pct in [0.5, 1.0, 2.0, 3.0, 5.0]:
    n_flags = int(len(y_val) * budget_pct / 100)
    # Flag the n_flags highest-scoring transactions
    cutoff = np.sort(val_probs)[-n_flags]
    preds_b = (val_probs >= cutoff).astype(int)
    tp_b = ((y_val.values == 1) & (preds_b == 1)).sum()
    fp_b = ((y_val.values == 0) & (preds_b == 1)).sum()
    rec_b = tp_b / (y_val.values == 1).sum()
    prec_b = tp_b / (tp_b + fp_b)
    print(f"{budget_pct:>10.1f}% | {cutoff:>7.3f} | {rec_b:>7.3f} | {prec_b:>9.3f} | {tp_b:>11,}")

# Running the Test Set

In [ ]:
raw = pd.read_parquet("data/train_data_merged.parquet").sort_values("TransactionDT").reset_index(drop = True)
n = len(raw)
tr = raw.iloc[:int(n*0.70)].copy()
va = raw.iloc[int(n*0.70):int(n*0.85)].copy()
te = raw.iloc[int(n*0.85):].copy()

state = fit_feature_pipeline(tr)
X_train, y_train = make_Xy(transform_features(tr,state), state)
X_val, y_val = make_Xy(transform_features(va,state), state)
X_test, y_test = make_Xy(transform_features(te,state), state)

xgb = joblib.load("models/xgb_baseline.pkl")
val_probs = xgb.predict_proba(X_val)[:,1]
print("Val PR-AUC:", round(average_precision_score(y_val, val_probs), 4), "(expect 0.5419)")

In [ ]:
test_probs = xgb.predict_proba(X_test)[:, 1]

print("=== XGBoost on SEALED TEST SET ===")
print("ROC-AUC:", round(roc_auc_score(y_test, test_probs), 4), " (val: 0.9080)")
print("PR-AUC :", round(average_precision_score(y_test, test_probs), 4), " (val: 0.5419)")

In [ ]:
test_amounts = te["TransactionAmt"].values

print(f"{'FlagBudget':>11} | {'Thresh':>7} | {'Recall':>7} | {'Precision':>9} | {'FraudCaught':>11}")
print("-" * 56)
for budget_pct in [0.5, 1.0, 2.0, 3.0, 5.0]:
    n_flags = int(len(y_test) * budget_pct / 100)
    cutoff = np.sort(test_probs)[-n_flags]
    preds_b = (test_probs >= cutoff).astype(int)
    tp_b = ((y_test.values == 1) & (preds_b == 1)).sum()
    fp_b = ((y_test.values == 0) & (preds_b == 1)).sum()
    rec_b = tp_b / (y_test.values == 1).sum()
    prec_b = tp_b / (tp_b + fp_b)
    print(f"{budget_pct:>10.1f}% | {cutoff:>7.3f} | {rec_b:>7.3f} | {prec_b:>9.3f} | {tp_b:>11,}")

In [ ]:
# --- Rebuild NN preprocessing (fit on train) ---
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_test_sc  = scaler.transform(X_test_imp)

# --- Logistic Regression on the full feature set ---
lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train_sc, y_train)
lr_test_probs = lr.predict_proba(X_test_sc)[:, 1]

# --- Neural net ---
class FraudMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x)

model = FraudMLP(X_train_sc.shape[1])
model.load_state_dict(torch.load("models/fraud_mlp.pt"))
model.eval()
with torch.no_grad():
    nn_test_probs = torch.sigmoid(
        model(torch.tensor(X_test_sc, dtype=torch.float32))
    ).numpy().ravel()

# --- Final comparison table ---
print(f"{'Model':<22} | {'ROC-AUC':>8} | {'PR-AUC':>8}")
print("-" * 44)
for name, p in [("Logistic Regression", lr_test_probs),
                ("Neural Net (MLP)", nn_test_probs),
                ("XGBoost", test_probs)]:
    print(f"{name:<22} | {roc_auc_score(y_test, p):>8.4f} | {average_precision_score(y_test, p):>8.4f}")

# SHAP

In [ ]:
raw = pd.read_parquet("data/train_data_merged.parquet").sort_values("TransactionDT").reset_index(drop = True)
n = len(raw)
tr = raw.iloc[:int(n*0.70)].copy()
va = raw.iloc[int(n*0.70):int(n*0.85)].copy()

state = fit_feature_pipeline(tr)
X_train, y_train = make_Xy(transform_features(tr, state), state)
X_val, y_val = make_Xy(transform_features(va, state), state)

xgb = joblib.load("models/xgb_baseline.pkl")
print("Val PR-AUC:", round(average_precision_score(y_val, xgb.predict_proba(X_val)[:,1]), 4), "(expect 0.5419)")

In [ ]:
# Tree explainer for tree model
explainer = shap.TreeExplainer(xgb)

# SHAP on full val set using a sample
sample = X_val.sample(n = 5000, random_state= 42)
shap_values = explainer.shap_values(sample)

print("Computed SHAP values. Shape", shap_values.shape)

In [ ]:
# Global importance = mean absolute SHAP value per feature
importance = np.abs(shap_values).mean(axis=0)
imp_df = pd.DataFrame({
    "feature": sample.columns,
    "mean_abs_shap": importance
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("Top 25 features by SHAP importance:")
print(imp_df.head(25).to_string(index=True))

# The visual summary (beeswarm) — top 20 features
shap.summary_plot(shap_values, sample, max_display=20, show=True)

In [ ]:
# Where did the engineered features rank overall (out of 426)?
engineered = ["amt_z_for_card", "card_amt_mean", "card_amt_std",
              "addr1_missing", "D6_missing", "D8_missing", "D12_missing",
              "dist2_missing", "id_31_missing",
              "ProductCD_C", "ProductCD_R", "card6_credit", "card6_debit",
              "card4_discover", "hour"]

print("Engineered feature rankings (out of 426):")
for f in engineered:
    if f in imp_df["feature"].values:
        rank = imp_df[imp_df["feature"] == f].index[0]
        val = imp_df[imp_df["feature"] == f]["mean_abs_shap"].values[0]
        print(f"  {f:<18} rank {rank+1:>3} | SHAP {val:.4f}")

Engineered feature rankings (out of 426):


NameError: name 'imp_df' is not defined